# 05 - Optimizacion MongoDB
Objetivo: ejecutar explain executionStats antes/despues.

In [ ]:
import os, json
from pymongo import MongoClient
from dotenv import load_dotenv
load_dotenv()
client = MongoClient(os.environ['MONGODB_URI'])
db = client[os.getenv('MONGODB_DATABASE', 'ecommify')]

In [ ]:
pipeline = [
    {'$match': {'status': 'delivered'}},
    {'$lookup': {'from': 'order_reviews', 'localField': 'order_id', 'foreignField': 'order_id', 'as': 'reviews'}},
    {'$unwind': {'path': '$reviews', 'preserveNullAndEmptyArrays': True}},
    {'$addFields': {'review_score_safe': {'$ifNull': ['$reviews.review_score', 0]}}},
    {'$group': {'_id': '$customer.state', 'total_orders': {'$sum': 1}, 'total_revenue': {'$sum': '$payment_summary.total_value'}, 'avg_score': {'$avg': '$review_score_safe'}}},
    {'$sort': {'total_revenue': -1}},
    {'$limit': 10}
]
explain = db.command('aggregate', 'orders_analytics', pipeline=pipeline, explain=True, allowDiskUse=True)
explain